# OASIS-1 Multitask MRI Demo

This notebook evaluates the small public-data reproduction demo:

- Input: single T1 MRI per subject
- Segmentation target: FSL FAST tissue labels
- Classification target: `CDR == 0` vs `CDR > 0`
- Cognitive target: MMSE

This is a pipeline proof on a 16-subject OASIS-1 subset, not a scientific performance claim.

In [ ]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, mean_absolute_error, mean_squared_error
from torch.utils.data import DataLoader

PROJECT = Path.cwd()
if not (PROJECT / "src").exists():
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

from src.data import MriMultitaskDataset
from src.metrics import dice_score
from src.train_multitask import LitMultitask, load_config

CONFIG_PATH = PROJECT / "configs" / "oasis1_demo_multitask.yaml"
CHECKPOINT_PATH = PROJECT / "models" / "oasis1_demo_multitask.ckpt"

config = load_config(CONFIG_PATH)
config, CHECKPOINT_PATH.exists()

## Dataset Manifest

In [ ]:
manifest = pd.read_csv(config["manifest"])
display(manifest[["subject_id", "age", "sex", "mmse", "cdr", "label"]])

summary = pd.DataFrame({
    "n_subjects": [len(manifest)],
    "controls_cdr0": [(manifest["label"] == 0).sum()],
    "impaired_cdr_gt0": [(manifest["label"] == 1).sum()],
    "mmse_min": [manifest["mmse"].min()],
    "mmse_max": [manifest["mmse"].max()],
})
display(summary)

## Load Model And Predict

In [ ]:
dataset = MriMultitaskDataset(
    manifest_path=config["manifest"],
    segmentation_root=config["segmentation_root"],
    target_shape=tuple(config["target_shape"]),
    cognitive_target=config["cognitive_target"],
    label_column=config["label_column"],
    require_segmentation=True,
)
loader = DataLoader(dataset, batch_size=1, shuffle=False)

model = LitMultitask.load_from_checkpoint(CHECKPOINT_PATH, config=config, map_location="cpu")
model.eval()

rows = []
dice_values = []
with torch.no_grad():
    for batch in loader:
        outputs = model(batch["image"])
        prob = torch.sigmoid(outputs["diagnosis_logit"]).view(-1).item()
        pred_label = int(prob >= 0.5)
        pred_mmse = outputs["cognition"].view(-1).item()
        dice = dice_score(outputs["segmentation"], batch["segmentation"]).item()
        dice_values.append(dice)
        rows.append({
            "subject_id": batch["subject_id"][0],
            "true_label": int(batch["label"].item()),
            "pred_label": pred_label,
            "dementia_probability": prob,
            "true_mmse": float(batch["cognition"].item()),
            "pred_mmse": pred_mmse,
            "seg_dice": dice,
        })

predictions = pd.DataFrame(rows)
display(predictions)

## Metrics

In [ ]:
y_true = predictions["true_label"]
y_pred = predictions["pred_label"]
mmse_true = predictions["true_mmse"]
mmse_pred = predictions["pred_mmse"]

metrics = pd.DataFrame({
    "accuracy": [accuracy_score(y_true, y_pred)],
    "balanced_accuracy": [balanced_accuracy_score(y_true, y_pred)],
    "mmse_mae": [mean_absolute_error(mmse_true, mmse_pred)],
    "mmse_rmse": [mean_squared_error(mmse_true, mmse_pred) ** 0.5],
    "mean_seg_dice": [float(np.mean(dice_values))],
})
display(metrics)

cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1], labels=["Pred CDR=0", "Pred CDR>0"])
ax.set_yticks([0, 1], labels=["True CDR=0", "True CDR>0"])
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center", color="black")
ax.set_title("Diagnosis Confusion Matrix")
plt.tight_layout()
plt.show()

## Example MRI And FAST Segmentation Slices

In [ ]:
def show_subject(row_index: int = 0) -> None:
    row = manifest.iloc[row_index]
    image = nib.load(row["mri_path"]).get_fdata()
    seg = nib.load(row["segmentation_path"]).get_fdata()
    z = image.shape[2] // 2

    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    axes[0].imshow(np.rot90(image[:, :, z]), cmap="gray")
    axes[0].set_title(f"T1 MRI: {row['subject_id']}")
    axes[0].axis("off")

    axes[1].imshow(np.rot90(seg[:, :, z]), cmap="viridis", vmin=0, vmax=3)
    axes[1].set_title("FAST tissue segmentation")
    axes[1].axis("off")
    plt.tight_layout()
    plt.show()

show_subject(0)
show_subject(8)

## Interpretation

This notebook demonstrates that the reproduction scaffold can run end-to-end on public substitute data. The current 16-subject OASIS-1 run is intentionally tiny and downsampled, so its metrics are only a technical sanity check. The next scientific step is to scale to more OASIS subjects, use a fixed train/validation/test split, and compare against baselines.